# 행동 데이터 기반 개인화 공고 탐색 프로토타입

기존 팀 프로젝트에서 확인한 사용자 행동 결과를 후속 미니분석으로 확장한다.

- **개인화 기준:** 최근 30일 검색·필터·지원 행동으로 관심 직무 결정
- **직무 결정 우선순위:** 실제 지원 직무 → 탐색 행동 기반 직무 추정
- **추천 후보:** 사용자의 관심 직무와 동일한 `job_field`
- **재정렬 기준:** 팀 프로젝트에서 상대적으로 높은 지원율을 보였던 **원격근무 가능 / 연봉 공개 공고** 우선
- **LLM 역할:** 추천 공고를 결정하지 않고, 최종 추천 이유만 자연어로 설명

> 북마크는 지원과 연관성이 있었지만 어떤 공고를 북마크했는지 식별할 수 없어 콘텐츠 신호로 사용하지 않는다.  
> 기존 세그먼트 역시 사용자의 행동 상태를 구분하는 데는 유용하지만, 어떤 공고를 추천할지 직접 결정하는 신호로는 사용하지 않는다.


# 1. 데이터 준비 및 분석 기준 설정

## 1-1. 데이터 및 사용자 기준

분석에 사용하는 데이터는 사용자 로그, 지원 이력, 공고 정보이다.

- 전체 사용자: **15,367명**
- 최근 30일 기준 개인화 신호 사용자: **7,064명 (45.97%)**
- 개인화 신호가 없는 사용자: **8,303명 (54.03%)**

| 사용자 유형 | 사용자 수 | 전체 비율 |
|---|---:|---:|
| 검색/필터 O + 지원직무 O | 2,389 | 15.55% |
| 검색/필터 O + 지원직무 X | 3,618 | 23.54% |
| 검색/필터 X + 지원직무 O | 1,057 | 6.88% |
| 둘 다 X | 8,303 | 54.03% |

## 1-2. 최근 30일을 선택한 이유

사용자별 **마지막 구직 관련 활동일**을 기준일로 사용한다.

`마지막 구직 활동일 = max(마지막 공고 탐색일, 마지막 지원일)`

| 기간 | 개인화 신호 사용자 | 전체 대비 |
|---|---:|---:|
| 14일 | 6,587 | 42.86% |
| **30일** | **7,064** | **45.97%** |
| 60일 | 7,505 | 48.84% |

- 14일은 최근성은 높지만 활용 가능한 사용자가 상대적으로 적다.
- 60일은 사용자 비율이 48.84%로 늘지만, 30일 대비 증가폭은 **2.87%p**에 그친다.
- 기간이 길어질수록 과거 관심사가 현재 관심사와 섞일 가능성이 커진다.

따라서 **최근성과 데이터 확보 수준의 균형을 고려해 30일을 최종 기준으로 채택**한다.


In [ ]:
import pandas as pd
from urllib.parse import urlparse, parse_qs
from sklearn.model_selection import train_test_split
from google.colab import drive

drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/코드잇 1팀 중급 프로젝트'

application_df = pd.read_csv(f'{BASE_PATH}/applicaion_2023.csv')
log_2023_df = pd.read_csv(f'{BASE_PATH}/log2023_df_url.csv')
job_2023_df = pd.read_csv(f'{BASE_PATH}/job_cdate_2023.csv')

print('application_df:', application_df.shape)
print('log_2023_df:', log_2023_df.shape)
print('job_2023_df:', job_2023_df.shape)


In [ ]:
# 로그 URL 파싱
log_df = log_2023_df.copy()
log_df['URL'] = log_df['URL'].fillna('').astype(str)

parsed = log_df['URL'].apply(urlparse)
log_df['path'] = parsed.apply(lambda x: x.path)
log_df['query_dict'] = parsed.apply(lambda x: parse_qs(x.query))

# 채용공고 탐색 로그 추출
job_search_df = log_df[
    log_df['path'] == 'api/jobs/job_title'
].copy()

features = [
    'keywords', 'specialty', 'location', 'tag',
    'remote', 'salary', 'language'
]

for feature in features:
    job_search_df[feature] = (
        job_search_df['query_dict']
        .apply(lambda x: x.get(feature, [None])[0])
    )

# 텍스트 탐색값 정규화
for col in ['keywords', 'specialty', 'location', 'tag']:
    job_search_df[col] = (
        job_search_df[col]
        .astype('string')
        .str.strip()
        .str.lower()
    )

# 날짜 형식 변환
log_df['date'] = pd.to_datetime(log_df['date'], errors='coerce')
job_search_df['date'] = pd.to_datetime(job_search_df['date'], errors='coerce')
application_df['cdate'] = pd.to_datetime(application_df['cdate'], errors='coerce')
job_2023_df['cdate'] = pd.to_datetime(job_2023_df['cdate'], errors='coerce')

# 전체 사용자
all_user_ids = pd.concat([
    log_df['user_uuid'],
    application_df['user_uuid']
]).dropna().drop_duplicates()

user_profile = pd.DataFrame({'user_uuid': all_user_ids})

print("전체 사용자 수:", len(user_profile))
print("채용공고 관련 로그 수:", len(job_search_df))


In [ ]:
# 지원 데이터와 공고 속성 연결
job_cols = [
    'job_uuid',
    'job_field',
    'allow_remote',
    'can_show_salary'
]

application_job_df = application_df.merge(
    job_2023_df[job_cols],
    on='job_uuid',
    how='left'
)

# 사용자별 마지막 탐색일 / 마지막 지원일
last_search = (
    job_search_df
    .groupby('user_uuid')['date']
    .max()
    .reset_index(name='last_search_date')
)

last_apply = (
    application_df
    .groupby('user_uuid')['cdate']
    .max()
    .reset_index(name='last_apply_date')
)

last_activity = (
    user_profile[['user_uuid']]
    .merge(last_search, on='user_uuid', how='left')
    .merge(last_apply, on='user_uuid', how='left')
)

last_activity['last_job_activity_date'] = (
    last_activity[['last_search_date', 'last_apply_date']]
    .max(axis=1)
)

def get_recent_search(n_days):
    temp = job_search_df.merge(
        last_activity[['user_uuid', 'last_job_activity_date']],
        on='user_uuid',
        how='left'
    )

    return temp[
        (temp['date'] <= temp['last_job_activity_date']) &
        (
            temp['date'] >=
            temp['last_job_activity_date'] - pd.Timedelta(days=n_days)
        )
    ].copy()

def get_recent_application(n_days):
    temp = application_job_df.merge(
        last_activity[['user_uuid', 'last_job_activity_date']],
        on='user_uuid',
        how='left'
    )

    return temp[
        (temp['cdate'] <= temp['last_job_activity_date']) &
        (
            temp['cdate'] >=
            temp['last_job_activity_date'] - pd.Timedelta(days=n_days)
        )
    ].copy()


In [ ]:
# 14 / 30 / 60일 비교
window_results = []

for n_days in [14, 30, 60]:
    search_n = get_recent_search(n_days)
    app_n = get_recent_application(n_days)

    search_users = set(
        search_n[
            search_n[['keywords', 'specialty', 'location', 'tag']]
            .notna()
            .any(axis=1)
        ]['user_uuid']
    )

    applied_users = set(
        app_n
        .dropna(subset=['job_field'])
        ['user_uuid']
    )

    personalization_users = search_users | applied_users

    window_results.append({
        '기간': f'{n_days}일',
        '개인화신호_사용자': len(personalization_users),
        '전체대비_비율': round(
            len(personalization_users) / len(user_profile) * 100,
            2
        )
    })

window_comparison = pd.DataFrame(window_results)
display(window_comparison)


In [ ]:
# 최종 기준: 최근 30일
N_DAYS = 30
recent_search = get_recent_search(N_DAYS)
recent_app = get_recent_application(N_DAYS)

recent_search_users = set(
    recent_search[
        recent_search[['keywords', 'specialty', 'location', 'tag']]
        .notna()
        .any(axis=1)
    ]['user_uuid']
)

recent_applied_users = set(
    recent_app
    .dropna(subset=['job_field'])
    ['user_uuid']
)

type_rows = []

for user_id in user_profile['user_uuid']:
    has_search = user_id in recent_search_users
    has_apply = user_id in recent_applied_users

    if has_search and has_apply:
        user_type = '검색/필터 O + 지원직무 O'
    elif has_search:
        user_type = '검색/필터 O + 지원직무 X'
    elif has_apply:
        user_type = '검색/필터 X + 지원직무 O'
    else:
        user_type = '둘 다 X'

    type_rows.append({
        'user_uuid': user_id,
        'user_type': user_type
    })

recent_user_type = pd.DataFrame(type_rows)

type_summary = (
    recent_user_type
    .groupby('user_type')
    .size()
    .reset_index(name='사용자 수')
)

type_summary['비율'] = (
    type_summary['사용자 수']
    / len(user_profile)
    * 100
).round(2)

display(type_summary)


In [ ]:
# 최근 30일 사용자 프로필 생성
def get_top_values(df, column, top_n=3):
    temp = (
        df
        .dropna(subset=[column])
        .groupby(['user_uuid', column])
        .size()
        .reset_index(name='count')
        .sort_values(
            ['user_uuid', 'count'],
            ascending=[True, False]
        )
    )

    return (
        temp
        .groupby('user_uuid')
        .head(top_n)
        .groupby('user_uuid')[column]
        .apply(list)
        .reset_index()
        .rename(columns={column: f'recent_{column}'})
    )

recent_keywords = get_top_values(recent_search, 'keywords')
recent_specialty = get_top_values(recent_search, 'specialty')
recent_location = get_top_values(recent_search, 'location')
recent_tag = get_top_values(recent_search, 'tag')

recent_applied_job_fields = (
    recent_app
    .dropna(subset=['job_field'])
    .groupby(['user_uuid', 'job_field'])
    .size()
    .reset_index(name='count')
    .sort_values(
        ['user_uuid', 'count'],
        ascending=[True, False]
    )
    .groupby('user_uuid')
    .head(3)
    .groupby('user_uuid')['job_field']
    .apply(list)
    .reset_index(name='recent_applied_job_fields')
)

recent_application_count = (
    recent_app
    .groupby('user_uuid')
    .size()
    .reset_index(name='recent_application_count')
)

recent_apply_attr = (
    recent_app
    .groupby('user_uuid')
    .agg(
        applied_remote_rate=('allow_remote', 'mean'),
        applied_salary_open_rate=('can_show_salary', 'mean')
    )
    .reset_index()
)

recent_apply_attr['applied_remote_rate'] *= 100
recent_apply_attr['applied_salary_open_rate'] *= 100

recent_remote_users = set(
    recent_search.dropna(subset=['remote'])['user_uuid']
)
recent_salary_users = set(
    recent_search.dropna(subset=['salary'])['user_uuid']
)

final_profile = user_profile[['user_uuid']].drop_duplicates().copy()

for profile_df in [
    recent_keywords,
    recent_specialty,
    recent_location,
    recent_tag,
    recent_applied_job_fields,
    recent_application_count,
    recent_apply_attr
]:
    final_profile = final_profile.merge(
        profile_df,
        on='user_uuid',
        how='left'
    )

final_profile['used_remote_filter'] = (
    final_profile['user_uuid'].isin(recent_remote_users)
)
final_profile['used_salary_filter'] = (
    final_profile['user_uuid'].isin(recent_salary_users)
)
final_profile['recent_application_count'] = (
    final_profile['recent_application_count']
    .fillna(0)
    .astype(int)
)

final_profile = final_profile.merge(
    recent_user_type,
    on='user_uuid',
    how='left'
)

personalized_users = final_profile[
    final_profile['user_type'] != '둘 다 X'
].copy()

print("개인화 대상 사용자 수:", len(personalized_users))
print(
    "전체 대비 비율:",
    round(len(personalized_users) / len(final_profile) * 100, 2),
    "%"
)


# 2. 탐색 행동 기반 관심 직무 추정

지원 직무 정보가 없는 사용자에게 추천 후보를 만들기 위해 최근 탐색 행동과 실제 지원 직무의 관계를 학습한다.

## 2-1. 학습 기준

- 학습 대상: **검색/필터 O + 지원직무 O 사용자 2,389명**
- 실제 지원 직무를 정답(`main_job_field`)으로 사용
- 탐색 신호: `keywords`, `specialty`, `tag`
- `location`은 직무 자체를 직접 나타내는 신호가 아니므로 직무 추정에서는 제외
- Train/Test = **70/30**, `random_state=42`

## 2-2. 최소 등장 횟수 / 직무 일치 비율 기준

키워드가 특정 직무와 연결된다고 판단하기 위해 두 조건을 함께 사용한다.

- `min_total`: 탐색값 최소 등장 횟수
- `min_ratio`: 해당 탐색값이 대표 직무와 연결된 비율

주요 비교 결과:

| 기준 | Accuracy | Coverage |
|---|---:|---:|
| 5 / 0.6 | 89.09% | 47.28% |
| 5 / 0.7 | 90.36% | 46.30% |
| 5 / 0.8 | 91.95% | 45.05% |
| **10 / 0.8** | **93.51%** | **42.96%** |
| 20 / 0.8 | 94.06% | 39.89% |
| 30 / 0.8 | 95.32% | 38.77% |
| 50 / 0.9 | 96.30% | 33.89% |

기준을 강화할수록 Accuracy는 올라가지만 Coverage는 감소한다.

**10 / 0.8을 선택한 이유**
- 10 / 0.8 → 20 / 0.8로 강화해도 Accuracy는 **0.55%p**만 증가
- 반면 Coverage는 **3.07%p 감소**
- 따라서 정확도와 적용 범위의 균형을 고려해 **최소 10회 + 80% 일치**를 최종 기준으로 사용

참고로 모든 사용자를 가장 많은 직무인 **SW 개발**로만 예측한 Baseline Accuracy는 **71.97%**였다.

> 검증 시에는 반드시 **Train 데이터로만 매핑 규칙을 만들고 Test에서 평가**한다.  
> 검증이 끝난 뒤 실제 프로토타입용 최종 매핑은 전체 학습 가능 사용자를 이용해 다시 생성한다.


In [ ]:
# 탐색 + 실제 지원 직무가 모두 있는 사용자
mapping_users = final_profile[
    final_profile['user_type'] == '검색/필터 O + 지원직무 O'
].copy()

# 사용자별 대표 실제 지원 직무
main_applied_field = (
    recent_app
    .dropna(subset=['job_field'])
    .groupby(['user_uuid', 'job_field'])
    .size()
    .reset_index(name='apply_count')
    .sort_values(
        ['user_uuid', 'apply_count'],
        ascending=[True, False]
    )
    .drop_duplicates('user_uuid')
    [['user_uuid', 'job_field']]
    .rename(columns={'job_field': 'main_job_field'})
)

mapping_search = recent_search.merge(
    main_applied_field,
    on='user_uuid',
    how='inner'
)

mapping_user_ids = mapping_users['user_uuid'].unique()

train_users, test_users = train_test_split(
    mapping_user_ids,
    test_size=0.3,
    random_state=42
)

train_search = mapping_search[
    mapping_search['user_uuid'].isin(train_users)
].copy()

test_search = recent_search[
    recent_search['user_uuid'].isin(test_users)
].copy()

test_actual = main_applied_field[
    main_applied_field['user_uuid'].isin(test_users)
].copy()

print("매핑 학습용 사용자:", len(mapping_users))
print("Train:", len(train_users))
print("Test:", len(test_users))


In [ ]:
# 탐색값 → 대표 직무 매핑 생성 함수
def build_feature_mapping(
    df,
    feature,
    min_total=10,
    min_ratio=0.8
):
    feature_field = (
        df
        .dropna(subset=[feature, 'main_job_field'])
        .groupby([feature, 'main_job_field'])
        .size()
        .reset_index(name='count')
    )

    feature_field['total'] = (
        feature_field
        .groupby(feature)['count']
        .transform('sum')
    )

    feature_field['ratio'] = (
        feature_field['count']
        / feature_field['total']
    )

    mapping = (
        feature_field
        .sort_values(
            [feature, 'ratio', 'count'],
            ascending=[True, False, False]
        )
        .drop_duplicates(feature)
    )

    return mapping[
        (mapping['total'] >= min_total) &
        (mapping['ratio'] >= min_ratio)
    ].copy()


In [ ]:
# keywords 기준 threshold 비교
threshold_results = []

for min_total in [5, 10, 20, 30, 50]:
    for min_ratio in [0.6, 0.7, 0.8, 0.9]:

        valid_mapping = build_feature_mapping(
            train_search,
            'keywords',
            min_total=min_total,
            min_ratio=min_ratio
        )

        keyword_to_field = dict(
            zip(
                valid_mapping['keywords'],
                valid_mapping['main_job_field']
            )
        )

        temp = test_search[
            ['user_uuid', 'keywords']
        ].copy()

        temp['predicted_field'] = (
            temp['keywords']
            .map(keyword_to_field)
        )

        user_prediction = (
            temp
            .dropna(subset=['predicted_field'])
            .groupby(['user_uuid', 'predicted_field'])
            .size()
            .reset_index(name='vote')
            .sort_values(
                ['user_uuid', 'vote'],
                ascending=[True, False]
            )
            .drop_duplicates('user_uuid')
            [['user_uuid', 'predicted_field']]
        )

        evaluation = test_actual.merge(
            user_prediction,
            on='user_uuid',
            how='left'
        )

        predicted = evaluation.dropna(
            subset=['predicted_field']
        )

        accuracy = (
            (
                predicted['main_job_field']
                == predicted['predicted_field']
            ).mean()
            if len(predicted) > 0 else 0
        )

        coverage = (
            len(predicted) / len(evaluation)
            if len(evaluation) > 0 else 0
        )

        threshold_results.append({
            'min_total': min_total,
            'min_ratio': min_ratio,
            'accuracy_pct': round(accuracy * 100, 2),
            'coverage_pct': round(coverage * 100, 2),
            '예측사용자수': len(predicted),
            'test사용자수': len(evaluation)
        })

threshold_eval = pd.DataFrame(threshold_results)
display(threshold_eval)


In [ ]:
# Baseline: 모든 Test 사용자를 가장 빈도가 높은 직무로 예측
most_common_field = (
    main_applied_field[
        main_applied_field['user_uuid'].isin(train_users)
    ]['main_job_field']
    .mode()[0]
)

baseline_accuracy = (
    test_actual['main_job_field']
    == most_common_field
).mean()

print("가장 많은 직무:", most_common_field)
print(
    "Baseline Accuracy:",
    round(baseline_accuracy * 100, 2),
    "%"
)


In [ ]:
# 10 / 0.8 기준으로 Train에서 세 탐색 신호 매핑 생성
MIN_TOTAL = 10
MIN_RATIO = 0.8

train_keyword_mapping = build_feature_mapping(
    train_search, 'keywords', MIN_TOTAL, MIN_RATIO
)
train_specialty_mapping = build_feature_mapping(
    train_search, 'specialty', MIN_TOTAL, MIN_RATIO
)
train_tag_mapping = build_feature_mapping(
    train_search, 'tag', MIN_TOTAL, MIN_RATIO
)

train_maps = {
    'keywords': dict(zip(
        train_keyword_mapping['keywords'],
        train_keyword_mapping['main_job_field']
    )),
    'specialty': dict(zip(
        train_specialty_mapping['specialty'],
        train_specialty_mapping['main_job_field']
    )),
    'tag': dict(zip(
        train_tag_mapping['tag'],
        train_tag_mapping['main_job_field']
    ))
}

# Test 사용자 통합 예측
test_votes = []

for feature in ['keywords', 'specialty', 'tag']:
    temp = test_search[['user_uuid', feature]].copy()
    temp['predicted_field'] = temp[feature].map(
        train_maps[feature]
    )
    test_votes.append(
        temp[['user_uuid', 'predicted_field']]
    )

test_votes = pd.concat(
    test_votes,
    ignore_index=True
).dropna(subset=['predicted_field'])

test_prediction = (
    test_votes
    .groupby(['user_uuid', 'predicted_field'])
    .size()
    .reset_index(name='vote')
    .sort_values(
        ['user_uuid', 'vote'],
        ascending=[True, False]
    )
    .drop_duplicates('user_uuid')
)

evaluation = test_actual.merge(
    test_prediction[['user_uuid', 'predicted_field']],
    on='user_uuid',
    how='left'
)

predicted = evaluation.dropna(
    subset=['predicted_field']
)

combined_accuracy = (
    predicted['main_job_field']
    == predicted['predicted_field']
).mean()

combined_coverage = (
    len(predicted) / len(evaluation)
)

print(
    "통합 Accuracy:",
    round(combined_accuracy * 100, 2),
    "%"
)
print(
    "통합 Coverage:",
    round(combined_coverage * 100, 2),
    "%"
)
print(
    "예측 사용자:",
    len(predicted),
    "/",
    len(evaluation)
)


# 3. 최종 관심 직무 결정

최종 추천 직무는 다음 우선순위로 결정한다.

1. **최근 30일 실제 지원 직무가 있는 사용자**
   - 실제 지원 직무를 메인 신호로 사용
2. **지원 직무가 없는 사용자**
   - `keywords + specialty + tag`를 10 / 0.8 기준으로 직무와 매핑
   - 세 신호가 지지한 직무를 합쳐 **가장 많은 표를 받은 직무**를 최종 추정
3. **탐색 + 지원 정보가 모두 있는 사용자**
   - 실제 지원 직무를 우선 사용
   - 동시에 탐색→지원 직무 매핑 규칙을 만드는 학습 데이터로 활용

기존 실험 결과에서는:
- 실제 지원 직무: **3,446명**
- 탐색 기반 추정: **2,049명**
- 직무 추정 불가: **1,569명**
- 최종 직무 결정 사용자: **5,495명**
- 전체 사용자 대비: **35.76%**
- 개인화 신호 사용자 대비: **77.79%**

> 기존 노트북에서 기록된 통합 Accuracy 95.91%, Coverage 71.55%는 전체 매핑을 이용한 검증 단계가 섞여 있었으므로, GitHub 정리본에서는 위 2번 섹션의 Train-only 매핑 → Test 평가 결과를 최종 검증 수치로 사용한다.


In [ ]:
# 검증이 끝났으므로 프로토타입용 최종 매핑은 전체 학습 가능 사용자로 재학습
final_keyword_mapping = build_feature_mapping(
    mapping_search, 'keywords', MIN_TOTAL, MIN_RATIO
)
final_specialty_mapping = build_feature_mapping(
    mapping_search, 'specialty', MIN_TOTAL, MIN_RATIO
)
final_tag_mapping = build_feature_mapping(
    mapping_search, 'tag', MIN_TOTAL, MIN_RATIO
)

keyword_to_field = dict(zip(
    final_keyword_mapping['keywords'],
    final_keyword_mapping['main_job_field']
))
specialty_to_field = dict(zip(
    final_specialty_mapping['specialty'],
    final_specialty_mapping['main_job_field']
))
tag_to_field = dict(zip(
    final_tag_mapping['tag'],
    final_tag_mapping['main_job_field']
))


In [ ]:
# 지원 직무가 없는 탐색-only 사용자
search_only_users = final_profile[
    final_profile['user_type'] == '검색/필터 O + 지원직무 X'
].copy()

search_only_logs = recent_search[
    recent_search['user_uuid'].isin(
        search_only_users['user_uuid']
    )
][
    ['user_uuid', 'keywords', 'specialty', 'tag']
].copy()

search_only_logs['keyword_field'] = (
    search_only_logs['keywords']
    .map(keyword_to_field)
)
search_only_logs['specialty_field'] = (
    search_only_logs['specialty']
    .map(specialty_to_field)
)
search_only_logs['tag_field'] = (
    search_only_logs['tag']
    .map(tag_to_field)
)

field_votes = pd.concat([
    search_only_logs[
        ['user_uuid', 'keyword_field']
    ].rename(columns={'keyword_field': 'estimated_field'}),

    search_only_logs[
        ['user_uuid', 'specialty_field']
    ].rename(columns={'specialty_field': 'estimated_field'}),

    search_only_logs[
        ['user_uuid', 'tag_field']
    ].rename(columns={'tag_field': 'estimated_field'})
])

field_votes = field_votes.dropna(
    subset=['estimated_field']
)

estimated_job_field = (
    field_votes
    .groupby(['user_uuid', 'estimated_field'])
    .size()
    .reset_index(name='vote')
    .sort_values(
        ['user_uuid', 'vote'],
        ascending=[True, False]
    )
    .drop_duplicates('user_uuid')
)

print(
    "탐색 기반 직무 추정 사용자:",
    len(estimated_job_field)
)
print(
    "탐색-only 사용자 중 추정 가능 비율:",
    round(
        len(estimated_job_field)
        / len(search_only_users)
        * 100,
        2
    ),
    "%"
)


In [ ]:
# 실제 지원 직무 우선 → 없으면 탐색 기반 추정 직무 사용
personalized_users = personalized_users.merge(
    estimated_job_field[
        ['user_uuid', 'estimated_field']
    ],
    on='user_uuid',
    how='left'
)

personalized_users['actual_job_field'] = (
    personalized_users['recent_applied_job_fields']
    .apply(
        lambda x: x[0]
        if isinstance(x, list) and len(x) > 0
        else None
    )
)

personalized_users['final_job_field'] = (
    personalized_users['actual_job_field']
    .fillna(personalized_users['estimated_field'])
)

def get_field_source(row):
    if pd.notna(row['actual_job_field']):
        return '실제 지원직무'
    if pd.notna(row['estimated_field']):
        return '탐색기반 추정'
    return '직무 추정 불가'

personalized_users['job_field_source'] = (
    personalized_users.apply(
        get_field_source,
        axis=1
    )
)

display(
    personalized_users['job_field_source']
    .value_counts()
)

final_field_users = (
    personalized_users['final_job_field']
    .notna()
    .sum()
)

print("최종 직무 결정 사용자:", final_field_users)
print(
    "전체 사용자 대비:",
    round(final_field_users / len(final_profile) * 100, 2),
    "%"
)
print(
    "개인화 신호 사용자 대비:",
    round(
        final_field_users / len(personalized_users) * 100,
        2
    ),
    "%"
)


# 4. 공고 후보 생성 및 우선순위 재정렬

최종 관심 직무가 결정된 사용자에게 아래 순서로 추천 후보를 생성한다.

```text
최근 30일 사용자 행동
→ 관심 직무 결정
→ 같은 job_field 공고만 후보
→ 사용자 기준일 이후 생성된 미래 공고 제외
→ 이미 지원한 공고 제외
→ 원격근무 가능 / 연봉 공개 공고 우선
→ 동일 점수에서는 최신 공고 우선
→ 회사 중복 최소화
→ TOP 5
```

## 공고 속성 점수 `attribute_score`

팀 프로젝트에서 **원격근무 가능 공고와 연봉 공개 공고의 지원율이 상대적으로 높았던 결과**를 후속 추천 재정렬 규칙으로 활용한다.

| attribute_score | 원격근무 | 연봉 공개 | 우선순위 |
|---:|:---:|:---:|---|
| 2 | O | O | 최우선 |
| 1 | O | X | 우선 |
| 1 | X | O | 우선 |
| 0 | X | X | 후순위 |

이 점수는 **추천 정확도 점수**가 아니라, 팀 프로젝트에서 확인한 고지원율 공고 속성을 얼마나 만족하는지를 나타내는 **상단 배치 우선순위 점수**이다.


In [ ]:
# 최종 추천 대상 사용자
recommendation_users = personalized_users[
    personalized_users['final_job_field'].notna()
].copy()

recommendation_users = recommendation_users.merge(
    last_activity[
        ['user_uuid', 'last_job_activity_date']
    ],
    on='user_uuid',
    how='left'
)

# 사용자별 기존 지원 공고
applied_jobs_by_user = (
    application_df
    .groupby('user_uuid')['job_uuid']
    .apply(set)
    .to_dict()
)

print("최종 추천 대상 사용자:", len(recommendation_users))


In [ ]:
def recommend_top5(user_row):
    user_id = user_row['user_uuid']
    job_field = user_row['final_job_field']
    reference_date = pd.to_datetime(
        user_row['last_job_activity_date']
    )

    # 1. 관심 직무와 동일한 공고
    candidates = job_2023_df[
        job_2023_df['job_field'] == job_field
    ].copy()

    candidates['cdate'] = pd.to_datetime(
        candidates['cdate'],
        errors='coerce'
    )

    # 2. 기준일 당시 존재했던 공고만
    # 사용자 활동일이 일 단위이므로 공고 생성일도 날짜 단위로 비교
    candidates = candidates[
        candidates['cdate'].dt.normalize()
        <= reference_date.normalize()
    ].copy()

    # 3. 이미 지원한 공고 제외
    already_applied = applied_jobs_by_user.get(
        user_id,
        set()
    )

    candidates = candidates[
        ~candidates['job_uuid'].isin(already_applied)
    ].copy()

    # 4. 공고 속성 점수
    candidates['attribute_score'] = (
        candidates['allow_remote'].fillna(0).astype(int)
        +
        candidates['can_show_salary'].fillna(0).astype(int)
    )

    # 5. 속성 점수 우선 → 같은 점수면 최신 공고 우선
    candidates = candidates.sort_values(
        ['attribute_score', 'cdate'],
        ascending=[False, False]
    )

    # 6. 동일 회사 중복 최소화
    candidates = candidates.drop_duplicates(
        subset='company_uuid'
    )

    return candidates.head(5)


In [ ]:
# 전체 사용자 TOP5 생성
recommendation_list = []

for _, user in recommendation_users.iterrows():
    top5 = recommend_top5(user)

    if len(top5) == 0:
        continue

    top5 = top5.copy()
    top5['user_uuid'] = user['user_uuid']
    top5['final_job_field'] = user['final_job_field']
    top5['job_field_source'] = user['job_field_source']
    top5['recommend_rank'] = range(1, len(top5) + 1)

    recommendation_list.append(top5)

recommendation_df = pd.concat(
    recommendation_list,
    ignore_index=True
)

display(
    recommendation_df[
        [
            'user_uuid',
            'job_field_source',
            'final_job_field',
            'recommend_rank',
            'job_uuid',
            'company_uuid',
            'allow_remote',
            'can_show_salary',
            'attribute_score',
            'cdate'
        ]
    ].head(20)
)


# 5. 추천 결과 검증 및 사례 확인

## 5-1. 추천 생성 결과

최종 실행 결과:

- 직무 결정 사용자: **5,495명**
- 최소 1개 이상 추천 생성 사용자: **5,495명**
- 추천 생성률: **100%**
- 총 추천 공고 수: **27,458개**
- TOP5를 모두 받은 사용자: **5,488명 (99.87%)**
- 4개 추천: 2명
- 2개 추천: 5명

TOP5 미충족 7명을 확인한 결과:
- 대부분은 **기준일이 데이터 수집 초기 시점**이어서 당시 존재하는 동일 직무 공고 자체가 5개 미만
- 일부는 **기존 지원 공고 제외** 또는 **회사 중복 제거**로 후보가 추가 감소

미래 시점 공고를 억지로 사용하지 않고, **사용자 기준일 당시 실제 존재했던 공고만 추천**했기 때문에 발생한 정상적인 예외이다.

## 5-2. 재정렬 효과

동일한 사용자·직무·기준일 조건에서 단순 최신순 TOP5와 속성 점수 재정렬 TOP5를 비교했다.

| 지표 | 재정렬 전 | 재정렬 후 |
|---|---:|---:|
| 원격근무 가능 공고 비율 | 16.56% | 99.21% |
| 연봉 공개 공고 비율 | 30.09% | 99.22% |
| 원격 + 연봉공개 공고 비율 | 7.05% | 98.48% |

이는 **추천 정확도가 98.48%라는 의미가 아니다.**  
최종 TOP5에 노출된 추천 공고 중 **98.48%가 원격근무 가능 + 연봉 공개 조건을 모두 만족했다**는 의미이다.

`attribute_score` 분포:
- 2점: **27,041개 (98.48%)**
- 1점: **402개 (1.46%)**
- 0점: **15개 (0.05%)**

즉 팀 프로젝트에서 확인한 고지원율 공고 속성이 실제 추천 상단 노출에 강하게 반영되었음을 확인할 수 있다.


In [ ]:
# 추천 생성률 / TOP5 충족률
recommended_user_count = (
    recommendation_df['user_uuid']
    .nunique()
)

user_recommendation_count = (
    recommendation_df
    .groupby('user_uuid')
    .size()
)

recommendation_count_dist = (
    user_recommendation_count
    .value_counts()
    .sort_index()
)

full_top5_users = (
    user_recommendation_count == 5
).sum()

print("추천 생성 사용자:", recommended_user_count)
print(
    "추천 생성률:",
    round(
        recommended_user_count
        / len(recommendation_users)
        * 100,
        2
    ),
    "%"
)
print(
    "TOP5 완전 충족률:",
    round(
        full_top5_users
        / len(recommendation_users)
        * 100,
        2
    ),
    "%"
)
print("총 추천 공고 수:", len(recommendation_df))
display(recommendation_count_dist)


In [ ]:
# TOP5 미충족 사용자 원인 확인
less_than_5_ids = (
    user_recommendation_count[
        user_recommendation_count < 5
    ]
    .index
)

diagnostic_rows = []

for user_id in less_than_5_ids:
    user = recommendation_users[
        recommendation_users['user_uuid'] == user_id
    ].iloc[0]

    job_field = user['final_job_field']
    reference_date = pd.to_datetime(
        user['last_job_activity_date']
    )

    same_field = job_2023_df[
        job_2023_df['job_field'] == job_field
    ].copy()

    before_date = same_field[
        same_field['cdate'].dt.normalize()
        <= reference_date.normalize()
    ].copy()

    already_applied = applied_jobs_by_user.get(
        user_id,
        set()
    )

    after_apply = before_date[
        ~before_date['job_uuid'].isin(already_applied)
    ].copy()

    after_company = after_apply.drop_duplicates(
        subset='company_uuid'
    )

    diagnostic_rows.append({
        'user_uuid': user_id,
        'job_field': job_field,
        '기준일': reference_date.date(),
        '동일직무_전체공고': len(same_field),
        '기준일까지_공고': len(before_date),
        '지원제외_후': len(after_apply),
        '회사중복제거_후': len(after_company)
    })

diagnostic_df = pd.DataFrame(diagnostic_rows)
display(diagnostic_df)


In [ ]:
# 재정렬 전 Baseline: 동일 조건에서 최신 공고 우선
def recommend_top5_baseline(user_row):
    user_id = user_row['user_uuid']
    job_field = user_row['final_job_field']
    reference_date = pd.to_datetime(
        user_row['last_job_activity_date']
    )

    candidates = job_2023_df[
        job_2023_df['job_field'] == job_field
    ].copy()

    candidates = candidates[
        candidates['cdate'].dt.normalize()
        <= reference_date.normalize()
    ].copy()

    already_applied = applied_jobs_by_user.get(
        user_id,
        set()
    )

    candidates = candidates[
        ~candidates['job_uuid'].isin(already_applied)
    ].copy()

    candidates = candidates.sort_values(
        'cdate',
        ascending=False
    )

    candidates = candidates.drop_duplicates(
        subset='company_uuid'
    )

    return candidates.head(5)


baseline_list = []

for _, user in recommendation_users.iterrows():
    top5 = recommend_top5_baseline(user)

    if len(top5) == 0:
        continue

    top5 = top5.copy()
    top5['user_uuid'] = user['user_uuid']
    baseline_list.append(top5)

baseline_df = pd.concat(
    baseline_list,
    ignore_index=True
)

comparison = pd.DataFrame({
    'metric': [
        '원격근무 가능 공고 비율',
        '연봉 공개 공고 비율',
        '원격 + 연봉공개 공고 비율'
    ],
    '재정렬 전': [
        baseline_df['allow_remote'].fillna(0).mean() * 100,
        baseline_df['can_show_salary'].fillna(0).mean() * 100,
        (
            (baseline_df['allow_remote'].fillna(0) == 1) &
            (baseline_df['can_show_salary'].fillna(0) == 1)
        ).mean() * 100
    ],
    '재정렬 후': [
        recommendation_df['allow_remote'].fillna(0).mean() * 100,
        recommendation_df['can_show_salary'].fillna(0).mean() * 100,
        (
            (recommendation_df['allow_remote'].fillna(0) == 1) &
            (recommendation_df['can_show_salary'].fillna(0) == 1)
        ).mean() * 100
    ]
})

comparison[['재정렬 전', '재정렬 후']] = (
    comparison[['재정렬 전', '재정렬 후']]
    .round(2)
)

display(comparison)


In [ ]:
# attribute_score 분포
score_summary = (
    recommendation_df['attribute_score']
    .value_counts()
    .sort_index(ascending=False)
    .reset_index()
)

score_summary.columns = [
    'attribute_score',
    '추천 공고 수'
]

score_summary['비율'] = (
    score_summary['추천 공고 수']
    / len(recommendation_df)
    * 100
).round(2)

display(score_summary)


In [ ]:
# 실제 지원 직무 기반 대표 사례
actual_samples = (
    recommendation_users[
        recommendation_users['job_field_source']
        == '실제 지원직무'
    ]
    .sort_values('user_uuid')
    .drop_duplicates('final_job_field')
    .head(3)
)

display(
    actual_samples[
        [
            'user_uuid',
            'job_field_source',
            'final_job_field',
            'recent_keywords',
            'recent_specialty',
            'recent_tag'
        ]
    ]
)

actual_sample_result = recommendation_df[
    recommendation_df['user_uuid'].isin(
        actual_samples['user_uuid']
    )
].sort_values(
    ['user_uuid', 'recommend_rank']
)

display(
    actual_sample_result[
        [
            'user_uuid',
            'final_job_field',
            'recommend_rank',
            'job_uuid',
            'allow_remote',
            'can_show_salary',
            'attribute_score',
            'cdate'
        ]
    ]
)


In [ ]:
# 탐색 기반 직무 추정 대표 사례
estimated_samples = (
    recommendation_users[
        recommendation_users['job_field_source']
        == '탐색기반 추정'
    ]
    .sort_values('user_uuid')
    .drop_duplicates('final_job_field')
    .head(3)
)

display(
    estimated_samples[
        [
            'user_uuid',
            'job_field_source',
            'final_job_field',
            'recent_keywords',
            'recent_specialty',
            'recent_tag'
        ]
    ]
)

estimated_sample_result = recommendation_df[
    recommendation_df['user_uuid'].isin(
        estimated_samples['user_uuid']
    )
].sort_values(
    ['user_uuid', 'recommend_rank']
)

display(
    estimated_sample_result[
        [
            'user_uuid',
            'final_job_field',
            'recommend_rank',
            'job_uuid',
            'allow_remote',
            'can_show_salary',
            'attribute_score',
            'cdate'
        ]
    ]
)


### 추천 결과 인터페이스


In [ ]:
app_recommendation_df = recommendation_df[
    [
        'user_uuid',
        'job_field_source',
        'final_job_field',
        'recommend_rank',
        'job_uuid',
        'company_uuid',
        'job_field',
        'allow_remote',
        'can_show_salary',
        'attribute_score',
        'cdate'
    ]
].copy()

display(app_recommendation_df.head(20))

In [ ]:
app_recommendation_df = app_recommendation_df.rename(
    columns={
        'user_uuid': 'user_id',
        'job_uuid': 'job_id',
        'company_uuid': 'company_id',
        'final_job_field': 'interest_job_field',
        'recommend_rank': 'rank',
        'allow_remote': 'remote_available',
        'can_show_salary': 'salary_open'
    }
)

display(app_recommendation_df.head())

In [ ]:
app_recommendation_df['remote_text'] = (
    app_recommendation_df['remote_available']
    .map({
        1: '원격근무 가능',
        0: '원격근무 불가'
    })
)

app_recommendation_df['salary_text'] = (
    app_recommendation_df['salary_open']
    .map({
        1: '연봉 공개',
        0: '연봉 비공개'
    })
)

display(
    app_recommendation_df[
        [
            'user_id',
            'interest_job_field',
            'rank',
            'job_id',
            'remote_text',
            'salary_text'
        ]
    ].head(20)
)

In [ ]:
user_behavior_df = recommendation_users[
    [
        'user_uuid',
        'job_field_source',
        'recent_keywords',
        'recent_specialty',
        'recent_tag',
        'final_job_field'
    ]
].copy()

user_behavior_df = user_behavior_df.rename(
    columns={
        'user_uuid': 'user_id',
        'final_job_field': 'interest_job_field'
    }
)

display(user_behavior_df.head())

In [ ]:
# 앱용 사용자 행동 데이터 정리
user_behavior_app = user_behavior_df.copy()

# 1. NaN → "-"
cols = [
    'recent_keywords',
    'recent_specialty',
    'recent_tag'
]

user_behavior_app[cols] = (
    user_behavior_app[cols]
    .fillna('-')
)

# 2. 리스트 형태 → 보기 좋은 문자열
def clean_list_text(value):
    if isinstance(value, list):
        return ', '.join(map(str, value))

    if value == '-' or value == '':
        return '-'

    return str(value)

for col in cols:
    user_behavior_app[col] = (
        user_behavior_app[col]
        .apply(clean_list_text)
    )

# 3. 앱에서 사용할 컬럼만 유지
user_behavior_app = user_behavior_app[
    [
        'user_id',
        'job_field_source',
        'recent_keywords',
        'recent_specialty',
        'recent_tag',
        'interest_job_field'
    ]
]

display(user_behavior_app.head(10))

In [ ]:
app_recommendation_clean = app_recommendation_df.copy()

app_recommendation_clean['remote_text'] = (
    app_recommendation_clean['remote_available']
    .map({
        1: '가능',
        0: '불가'
    })
)

app_recommendation_clean['salary_text'] = (
    app_recommendation_clean['salary_open']
    .map({
        1: '공개',
        0: '비공개'
    })
)

app_recommendation_clean = (
    app_recommendation_clean[
        [
            'user_id',
            'interest_job_field',
            'rank',
            'job_id',
            'company_id',
            'job_field',
            'remote_text',
            'salary_text',
            'attribute_score',
            'cdate'
        ]
    ]
    .sort_values(
        ['user_id', 'rank']
    )
)

display(app_recommendation_clean.head(10))

In [ ]:
user_behavior_app.to_csv(
    'user_behavior_app.csv',
    index=False,
    encoding='utf-8-sig'
)

app_recommendation_clean.to_csv(
    'recommendation_app.csv',
    index=False,
    encoding='utf-8-sig'
)

print("앱용 데이터 저장 완료")